# Robustness Sweeps for Lag-Context Learning Rules

Companion to `hetero_rules_run.ipynb`, which introduces the update rule, the catalogue and
the crossover benchmark. Nothing is re-derived here.

## Contents

| § | sweep | axis |
|---|---|---|
| **1** | Corruption ratio | $\rho$, 0.05 - 0.35 |
| **2** | Network size | $N$, 20 - 400 |
| **2b** | Network size, $L_{\max}$-style measure | $N$, 50 - 500 |
| **2c** | Scaling $\omega$ with $N$ | $N \times \sigma_\omega$ grid |
| **3** | Superposition of learning rules | mixing weight $w$ |
| **4** | Crossover geometries | four branching topologies |
| **5** | Settle vs. loop | `boundary` |
| **6** | Frequency conditions | $\omega \equiv 0$ (displaced start), $\omega \equiv 1$ |
| **7** | Random (uncorrelated) patterns | -- |

Preceded by: *Configuration*, *The learning rules*, *The success metrics*, and a decoder /
duration check that fixes two methodological choices.

## What is measured

**Rules** -- the nine of `hetero_rules_run.ipynb`, all $d = 3$: `self`, `forward`, `skip-two`,
`self+forward`, `forward+skip2`, `bigram`, `multilag`, `coh bigram`, `coh trigram`.

**Topologies** -- a single smooth chain of `SEQ_LEN = 15` (16 patterns) for sweeps 1, 2, 3, 6;
independently random patterns for sweep 7; and four two-branch crossovers for sweep 4:

| tag | spec | $P$ | shared | $L$ |
|---|---|---|---|---|
| `2-1-2` | **(a)** eq (7) | 5 | $\{2\}$ | 1 |
| `2-2-2` | **(c)** eq (9) | 6 | $\{2,3\}$ | 2 |
| `2-3-2` | failure case | 7 | $\{2,3,4\}$ | 3 |
| `1-1-1` | legacy | 3 | $\{1\}$ | 1 |

Benchmark **(b)**, eq (8) -- three branches sharing one pattern -- is not implemented yet.
All chains are **smooth**: consecutive patterns differ by $\rho N$ flips, overlap $1 - 2\rho$.

**Metrics** -- for single-sequence retrieval:

1. **Retrieval fraction** (primary) -- the number of peaks correctly retrieved **in order
   before the first failure**, as a percentage of the sequence:
   $\texttt{compare\_to\_stored\_sequence(retrieved, seq)} / P$. This is
   `kuramoto_sequence.ipynb`'s measure. It gives partial credit, so a rule that recovers 14 of
   16 patterns reads as 88% rather than as total failure, and it resolves *how far* a rule
   gets when it cannot finish. Averaged over trials, reported with its standard deviation.
2. **Strict full-retrieval rate** (secondary) -- the fraction of trials scoring exactly 100%
   on metric 1. Kept in the printed tables; the plots show metric 1.
3. **Overlap quality** -- for a trial that retrieved *fully*, the mean over its patterns of
   each one's *peak* overlap, averaged over those trials only and reported with its standard
   deviation. A rule can be reliable and sloppy, or rare and crisp.

Crossover sweeps (§4, §5) keep the strict criterion: there the question is whether the
trajectory commits to the right branch, which has no natural partial credit.

`zeroed()` plots an undefined metric 3 (no fully-retrieved trial) as a literal `0.0`, so a
rule that fails reads as a flat zero line rather than a line that stops mid-axis.


### Which decoder, and why it matters at $P = 16$

Single-sequence retrieval is scored with the **peak decoder** (`decode_sequence_peaks`); the
streaming decoder (`decode_sequence_online`) is reported alongside it in sweep 1.

The streaming decoder logs a pattern only when it is the **global argmax**, above `tolerance`,
and ahead of the incumbent by `margin`. On a 16-pattern chain consecutive patterns overlap at
$1-2\rho$, so neighbouring overlaps sit within a few thousandths of each other:

```
A[14] peaks at 0.617  --  m[A13]=0.620 (argmax), m[A15]=0.607
A[8]  peaks at 0.565  --  above tolerance for 65 timesteps, argmax for 0
```

The path comes back as `[0, 1, ..., 13, 15]`: right order, one index missing, which the strict
criterion scores as total failure. Each of 15 transitions is a fresh chance, so it compounds
with length -- `bigram` scores 63% at `seq_len = 5` and 7% at 15.

**`margin` is not the lever.** It controls only the third condition; lowering it to zero moves
`bigram` 3% → 7%. The **argmax** condition is what blocks, and no margin relaxes it. **Time is
not the lever either**: outcomes are bit-identical from $T = 90$ to $T = 600$. Both are
measured below.

The peak decoder reads each pattern's own trace -- tallest hump above `tolerance`, ordered by
when it occurs -- and never compares patterns against each other, so it is immune. The rules
that still fail under it fail dynamically: `self` never traverses, `skip-two` skips by
construction, `coh trigram` decays below `tolerance` past position ~9.

The streaming decoder works best around $\rho \approx 0.15$ and degrades both ways -- low
$\rho$ because near-duplicates never separate, high $\rho$ because the drive fails. Sweep 1
shows this.


In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

import itertools

import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path


root_dir = Path.cwd().parent.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import kuramoto.kuramoto_library as krm
import kuramoto.hetero_learning.kuramoto_hetero as kc

## Configuration

`SEQ_LEN = 15` (16 patterns), three times `hetero_rules_run.ipynb`'s length. Integration time
scales with it -- `DWELL_PER_TRANSITION * seq_len`, i.e. 6.0 per transition, the same ratio
`CONFIG["T"] = 30.0` gave at `seq_len = 5`. Generous: outcomes are unchanged to $T = 600$.


In [ ]:
CONFIG = dict(
    dt=0.05,
    T=30.0,
    frequency_std=0.3,
    phase_noise=0.01,
    d=3,
    corruption_rate=0.2,
    tolerance=0.5,
    mode="ode",
)

N = 100

# Online-decoder hysteresis. Lowered from 0.01: on a P=16 mutation chain the gaps between
# neighbouring patterns' overlaps are ~0.003, an order of magnitude under the old value.
# It helps a little, but the margin is not what limits this decoder -- see the note below.
MARGIN = 0.001
SEED_POOL = 10_000
SINGLE_SEQ_TRIALS = 100

SEQ_LEN = 15                            # 16 stored patterns per single-sequence trial
DWELL_PER_TRANSITION = CONFIG["T"] / 5  # 6.0 -- the ratio hetero_rules_run.ipynb runs at

# Peak-decoder separation, in timesteps: one time unit. Only stops a single jittery hump
# being counted twice; the scores below are insensitive to it over 5-40 steps.
PEAK_MIN_DISTANCE = int(1.0 / CONFIG["dt"])

assert CONFIG["frequency_std"] > 0


## The learning rules

Unchanged from `hetero_rules_run.ipynb`. `GAIN_CATALOGUE` is the same nine through
`kc.proportional_gain`, which removes the $(1-2\rho)^{\sum \text{lags}}$ handicap a lag-heavy
gate carries. Sweeps 1, 2 and 2b run both -- at $P = 16$ that correction decides whether the
context rules retrieve at all.


In [ ]:
RAW_CATALOGUE = {
    "self":          kc.self_rule(),
    "forward":       kc.forward_rule(),
    "skip-two":      kc.skip_rule(nhop=2),
    "self+forward":  kc.mixture(kc.self_rule(weight=0.3), kc.forward_rule(weight=1.0)),
    "forward+skip2": kc.mixture(kc.forward_rule(weight=1.0), kc.skip_rule(2, weight=0.5)),
    "bigram":        kc.bigram_rule(),
    "multilag":      kc.multilag_rule((1.0, 1.0, 1.0)),
    "coh bigram":    kc.coherent_bigram_rule(),
    "coh trigram":   kc.coherent_trigram_rule(),
}

# The same rules through kc.proportional_gain, which divides each term by
# (1 - 2*rho)^(sum of its lags) and the rule by its total weight, so every rule has total
# gate 1 at its own operating point. The N sweeps below run both, because at this sequence
# length the difference between them is the whole result -- see section 2.
GAIN_CATALOGUE = {name: kc.proportional_gain(rule, CONFIG["corruption_rate"])
                  for name, rule in RAW_CATALOGUE.items()}


In [ ]:
def mean_overlap_score(result, seq):
    """METRIC 2 -- mean, over every non-cue pattern in `seq`, of that pattern's PEAK overlap,
    from an already-computed simulate() result. Only meaningful for a correctly retrieved
    trial -- callers must check that separately (see single_sequence_score_sweep)."""
    return float(result["overlaps"][:, 1:].max(axis=0).mean())


def zeroed(values):
    """nan -> 0.0.  `mean_overlap` is nan when a rule had no successful trial to average;
    plotting nan leaves a gap and the line ends abruptly mid-axis. A rule that never
    retrieves anything should read as a flat zero, which is what it scored."""
    return [0.0 if v != v else float(v) for v in values]


def band(ax, x, means, stds, **kw):
    """Draw metric 2 as a line with a +/- 1 standard-deviation band -- the spread of overlap
    quality across the successful trials."""
    means, stds = np.asarray(zeroed(means)), np.asarray(zeroed(stds))
    line, = ax.plot(x, means, "-o", **kw)
    ax.fill_between(x, means - stds, means + stds, color=line.get_color(), alpha=0.12, lw=0)
    return line


def generate_random_sequence(N, seq_len, name, seed):
    """A sequence of seq_len+1 INDEPENDENTLY random patterns -- no mutation chain, unlike
    generate_sequence, so consecutive patterns are unrelated rather than similar."""
    rng = np.random.default_rng(seed)
    patterns = [rng.choice([0.0, np.pi], size=N) for _ in range(seq_len + 1)]
    return krm.sequence_from_patterns(name, patterns)


def displaced_cue(seq, corruption_rate, phase_jitter, rng):
    """theta(0) for the omega = 0 runs: a displacement of the zeroth stored pattern.

    The displacement is NOT a stored pattern -- it is never added to `seq` and never
    memorised, it is only where the trajectory starts. Two parts:

      - `corruption_rate` neurons flipped 0 <-> pi, the same corruption operator the
        mutation chain is built from;
      - `phase_jitter`, a small uniform angular nudge on every neuron.

    The jitter is what makes the displacement effective. `corrupt_phase` keeps every phase
    on the {0, pi} lattice, where sin(theta) = 0 exactly -- so at omega = 0 the drive
    vanishes identically and a flip-only displacement is still an exact fixed point. The
    angular nudge takes theta off the lattice so sin(theta) != 0 and the dynamics can run."""
    theta = krm.corrupt_phase(seq[0], corruption_rate, rng)
    if phase_jitter > 0:
        theta = theta + rng.uniform(-phase_jitter, phase_jitter, size=theta.shape[0])
    return theta


def run_single_trial(net, seq, duration, boundary=None, cue_corruption_rate=0.0,
                     cue_phase_jitter=0.0, rng=None):
    """One integration, returning a simulate()-shaped result dict.

    With no displacement this is just `net.simulate`. With one, the cue is built by
    `displaced_cue` and handed to `net._integrate` directly, since `net.simulate` always
    constructs an exact, uncorrupted cue."""
    if cue_corruption_rate > 0 or cue_phase_jitter > 0:
        theta_init = displaced_cue(seq, cue_corruption_rate, cue_phase_jitter, rng)
        theta_history, time = net._integrate(seq, theta_init, None, None, duration, boundary)
        return {"time": time, "theta_history": theta_history, "cue": (seq.name, 0),
                "overlaps": net.overlaps(theta_history, seq)}
    return net.simulate(seq, T=duration, boundary=boundary)


def single_sequence_score_sweep(catalogue, seq_len, corruption_rate, N, trials=SINGLE_SEQ_TRIALS,
                                seed=0, boundary=None, config_overrides=None,
                                cue_corruption_rate=0.0, cue_phase_jitter=0.0,
                                random_patterns=False, duration=None, decoder="peaks"):
    """For each rule, run `trials` freshly generated single sequences of size `N` and report:
      - retrieval_fraction / fraction_std: METRIC 1, the mean over trials of
        `compare_to_stored_sequence(retrieved, seq) / len(seq)` -- the share of the sequence
        recovered, in order, before the first failure. Partial credit; 1.0 means full recall.
      - retrieval_rate: METRIC 2, the STRICT rate -- the fraction of trials scoring exactly
        1.0 on metric 1.
      - mean_overlap / overlap_std / overlap_var: METRIC 3, mean_overlap_score over ONLY the
        fully-retrieved trials, and its spread (nan if none -- plot via `zeroed`).
      - n_correct: how many trials the metric-3 statistics are averaged over.

    Decoded with `decode_sequence_peaks` by default -- see the note above on why the streaming
    decoder under-reports badly at seq_len = 15. Pass decoder="online" for the streaming one.

    duration: integration time. Defaults to DWELL_PER_TRANSITION * seq_len, so a longer
        sequence gets proportionally longer to traverse and the sweep measures the rule
        rather than the clock.
    boundary: passed straight through (None/"self" settles at the last pattern, "cycle"
        loops back to the first) -- used by the settle-vs-loop test.
    config_overrides: dict merged over CONFIG for this call (e.g. frequency_mean/std) --
        used by the frequency-condition tests.
    cue_corruption_rate / cue_phase_jitter: start from a displacement of seq[0] instead of
        seq[0] itself -- see `displaced_cue`. Needed for the omega = 0 configuration.
    random_patterns: if True, use generate_random_sequence instead of generate_sequence --
        stored patterns are independent rather than a mutation chain (corruption_rate is
        then unused).
    """
    duration = DWELL_PER_TRANSITION * seq_len if duration is None else duration
    rng = np.random.default_rng(seed)
    net_config = {**CONFIG, **(config_overrides or {})}
    results = {}
    for name, rule in catalogue.items():
        net = kc.ContextNetwork(rule=rule, N=N, seed=10, **net_config)
        overlap_vals, fractions = [], []
        for _ in range(trials):
            if random_patterns:
                seq = generate_random_sequence(N, seq_len, name="A", seed=int(rng.integers(SEED_POOL)))
            else:
                seq = krm.generate_sequence(N, seq_len, corruption_rate, name="A",
                                            seed=int(rng.integers(SEED_POOL)))
            result = run_single_trial(net, seq, duration, boundary=boundary,
                                      cue_corruption_rate=cue_corruption_rate,
                                      cue_phase_jitter=cue_phase_jitter, rng=rng)
            retrieved = (net.decode_sequence_peaks(result, seq, min_distance=PEAK_MIN_DISTANCE)
                         if decoder == "peaks"
                         else net.decode_sequence_online(result, seq, margin=MARGIN))
            reached = net.compare_to_stored_sequence(retrieved, seq)
            fractions.append(reached / len(seq))
            if reached == len(seq):
                overlap_vals.append(mean_overlap_score(result, seq))
        results[name] = dict(
            retrieval_fraction=float(np.mean(fractions)),
            fraction_std=float(np.std(fractions)),
            retrieval_rate=len(overlap_vals) / trials,
            n_correct=len(overlap_vals),
            mean_overlap=float(np.mean(overlap_vals)) if overlap_vals else float("nan"),
            overlap_std=float(np.std(overlap_vals)) if overlap_vals else float("nan"),
            overlap_var=float(np.var(overlap_vals)) if overlap_vals else float("nan"),
        )
    return results


---

## Is the sequence length or the clock the limit?

Two checks that fix the choices above. **Duration** is swept over almost an order of
magnitude; the outcomes come back bit-identical, because the trajectories hit the stationary
early-stop well inside the shortest duration. **`margin`** is scanned down to zero; it buys a
few points, because the binding condition is the argmax test, which no margin relaxes. What
moves the numbers is the decoder, by 20-90 points per rule.


In [ ]:
DURATION_CHECK = [90.0, 150.0, 250.0, 400.0, 600.0]
MARGIN_CHECK = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01]
DECODER_CHECK_RULES = ["forward", "bigram", "multilag", "coh bigram", "coh trigram"]


def score_with(decoder, rule, duration, trials=30, seed=0, margin=None):
    """Strict full-retrieval success rate for one rule, under a given decoder and duration."""
    net = kc.ContextNetwork(rule=RAW_CATALOGUE[rule], N=N, seed=10, **CONFIG)
    rng = np.random.default_rng(seed)
    n_correct = 0
    for _ in range(trials):
        seq = krm.generate_sequence(N, SEQ_LEN, CONFIG["corruption_rate"], name="A",
                                    seed=int(rng.integers(SEED_POOL)))
        result = net.simulate(seq, T=duration)
        retrieved = (net.decode_sequence_online(result, seq,
                                               margin=MARGIN if margin is None else margin)
                     if decoder == "online"
                     else net.decode_sequence_peaks(result, seq, min_distance=PEAK_MIN_DISTANCE))
        n_correct += net.compare_to_stored_sequence(retrieved, seq) == len(seq)
    return n_correct / trials


print(f"Strict full-retrieval success at seq_len={SEQ_LEN}, 30 seeds -- duration vs decoder")
print(f"{'rule':<14}{'decoder':<9}" + "".join(f"T={t:<8g}" for t in DURATION_CHECK))
for rule in DECODER_CHECK_RULES:
    for decoder in ("online", "peaks"):
        row = "".join(f"{score_with(decoder, rule, t):<10.0%}" for t in DURATION_CHECK)
        print(f"{rule if decoder == 'online' else '':<14}{decoder:<9}{row}")

print()
print(f"Streaming decoder vs margin (T={DURATION_CHECK[0]:g}, 30 seeds)")
print(f"{'rule':<14}" + "".join(f"m={m:<9g}" for m in MARGIN_CHECK) + f"{'peaks':>9}")
for rule in DECODER_CHECK_RULES:
    row = "".join(f"{score_with('online', rule, DURATION_CHECK[0], margin=m):<11.0%}"
                  for m in MARGIN_CHECK)
    print(f"{rule:<14}{row}{score_with('peaks', rule, DURATION_CHECK[0]):>9.0%}")


---

# 1. Corruption-ratio sweep

Single smooth chain, `SEQ_LEN = 15`. Metric 1 under **both** decoders, metric 2 under the
peak decoder, for **both** catalogues.

The swept axis is the corruption ratio $\rho$ itself -- the fraction of neurons flipped per
mutation step, which sets how similar consecutive patterns are ($1 - 2\rho$ exactly). It is
the parameter; the absolute Hamming distance $\rho N$ is whatever $N$ makes it.

The gain catalogue is rebuilt **at each $\rho$**, since `kc.proportional_gain(rule, rho)`
divides by $(1-2\rho)^{\sum\text{lags}}$ -- reusing one built at $\rho = 0.2$ would misstate
every other column.

Two failure modes bracket the curve. **Low $\rho$**: near-duplicate patterns the decoder
cannot separate, which hits the streaming decoder hardest. **High $\rho$**: patterns too far
apart for the drive to bridge; past $\rho \approx 0.25$ nothing completes a 16-pattern chain.
So the breakdown point is reported as the **largest** $\rho$ with success $\ge 50\%$, scanning
from the high end -- the first crossing can land on the low-$\rho$ failure instead.


In [ ]:
CORRUPTION_SWEEP_RATES = np.linspace(0.05, 0.35, 13)

corruption_results = {}
for tag in ("raw", "gain"):
    corruption_results[tag] = {}
    for rho in CORRUPTION_SWEEP_RATES:
        # the gain correction depends on rho, so rebuild the catalogue at each operating point
        catalogue = (RAW_CATALOGUE if tag == "raw" else
                     {name: kc.proportional_gain(rule, rho) for name, rule in RAW_CATALOGUE.items()})
        corruption_results[tag][rho] = {
            dec: single_sequence_score_sweep(catalogue, seq_len=SEQ_LEN, corruption_rate=rho,
                                             N=N, trials=SINGLE_SEQ_TRIALS, decoder=dec)
            for dec in ("peaks", "online")
        }

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
for row, tag in enumerate(("raw", "gain")):
    res = corruption_results[tag][CORRUPTION_SWEEP_RATES[0]]  # for key order only
    for name in RAW_CATALOGUE:
        for col, dec in enumerate(("peaks", "online")):
            band(axes[row, col], CORRUPTION_SWEEP_RATES,
                 [corruption_results[tag][r][dec][name]["retrieval_fraction"]
                  for r in CORRUPTION_SWEEP_RATES],
                 [corruption_results[tag][r][dec][name]["fraction_std"]
                  for r in CORRUPTION_SWEEP_RATES], label=name)
        band(axes[row, 2], CORRUPTION_SWEEP_RATES,
             [corruption_results[tag][r]["peaks"][name]["mean_overlap"] for r in CORRUPTION_SWEEP_RATES],
             [corruption_results[tag][r]["peaks"][name]["overlap_std"] for r in CORRUPTION_SWEEP_RATES],
             label=name)
    axes[row, 0].axhline(0.5, color="k", ls="--", lw=1)
    axes[row, 0].set_ylabel("retrieval fraction")
    axes[row, 0].set_title(f"Metric 1, peak decoder -- {tag}")
    axes[row, 1].axhline(0.5, color="k", ls="--", lw=1)
    axes[row, 1].set_ylabel("retrieval fraction")
    axes[row, 1].set_title(f"Metric 1, streaming decoder (margin={MARGIN}) -- {tag}")
    axes[row, 2].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1)
    axes[row, 2].set_ylabel("mean overlap (fully-retrieved trials; 0 = none)")
    axes[row, 2].set_title(f"Metric 3, peak decoder -- {tag} (band = $\\pm 1$ s.d.)")
    for ax in axes[row]:
        ax.set_xlabel(r"corruption ratio $\rho$  (consecutive-pattern overlap $= 1 - 2\rho$)")
        ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
fig.suptitle(f"1. Corruption-ratio sweep -- single smooth chain, P={SEQ_LEN + 1}, N={N}")
plt.tight_layout(); plt.show()

for tag in ("raw", "gain"):
    print(f"-- {tag} --")
    print(f"{'rule':<15}{'largest rho with fraction >= 90% (peaks/online)':>48}"
          f"{'strict rate at rho=0.20':>26}")
    for name in RAW_CATALOGUE:
        best = {}
        for dec in ("peaks", "online"):
            passing = [r for r in CORRUPTION_SWEEP_RATES
                       if corruption_results[tag][r][dec][name]["retrieval_fraction"] >= 0.9]
            best[dec] = f"{max(passing):.3f}" if passing else "never"
        near = min(CORRUPTION_SWEEP_RATES, key=lambda r: abs(r - 0.20))
        strict = corruption_results[tag][near]["peaks"][name]["retrieval_rate"]
        print(f"{name:<15}{best['peaks'] + ' / ' + best['online']:>48}{strict:>26.0%}")
    print()


# 2. Network-size sweep

Single smooth chain, `SEQ_LEN = 15`. Metrics 1 and 3, **both** catalogues.

`corruption_rate` is held fixed at `CONFIG["corruption_rate"]` across every `N` tested, so
absolute Hamming distance between consecutive patterns grows with `N`
(`= corruption_rate * N`) -- this asks whether a bigger network makes the same *relative*
task (same fraction of neurons flipped per step) easier or harder, not whether it can
tolerate the same absolute perturbation.


**It does not: the raw context rules get worse.** `bigram` scores 80% at $N=50$, 65% at
$N=100$, 0% from $N=200$ on; `coh bigram` and `coh trigram` do the same. Measured, and ruled
out as explanations:

- *Not the decoder* -- identical shape under both.
- *Not chain-to-chain variance* -- the per-chain distribution is unimodal and shifts as a
  whole: at $N=50$ every chain sustains 0.57-0.72, at $N=800$ every chain sustains 0.32-0.56.
- *Not noise-assisted escape* -- injecting phase noise at $N=400$ only makes it worse
  (mean fraction 0.58 → 0.25 as `phase_noise` goes 0 → 0.4).
- *Not a peak/extreme-value artefact* -- the overlap traces are smooth (temporal s.d. ~0.01).
- *Not the `tolerance` threshold* -- dropping it to 0.3 still leaves $N=200$ at 10%.

What changes is the **per-step transfer**. Mean peak overlap by position, raw `bigram`:

| $N$ | position 1 → 15 | |
|---|---|---|
| 50 | 0.80 0.73 0.69 0.69 0.66 ... 0.64 0.64 0.70 | plateaus at ≈0.65 |
| 100 | 0.78 0.72 0.68 0.68 0.65 ... 0.55 0.54 0.62 | plateaus at ≈0.55 |
| 200 | 0.76 0.68 0.65 0.62 0.59 ... 0.27 0.22 0.20 | decays geometrically |
| 400 | 0.76 0.70 0.65 0.62 0.59 ... 0.30 0.24 0.17 | decays geometrically |

Below $N \approx 150$ the cascade has a stable non-zero fixed point and self-sustains; above
it that fixed point is gone and the overlap decays toward noise.

**This is specific to the configured `frequency_std = 0.3`.** §2c varies $N$ and
$\sigma_\omega$ independently and finds that at $\sigma_\omega \le 0.2$ the $N$-dependence
disappears entirely -- every rule scores 1.00 at every $N \ge 100$. The raw rules sit at a
*critical disorder* between 0.2 and 0.3, and 0.3 falls just inside it at small $N$ and just
outside at large $N$. So $N$ is not the control parameter; read the curves below together
with §2c.

`kc.proportional_gain` multiplies the gate by $(1-2\rho)^{-2} = 2.78$, which puts it on the
sustaining side at every $N$: plateau ≈0.74, flat in position, **100% from $N = 100$ on**. So
the raw curves mean "under-driven at this length", not "cannot store 16 patterns". Note this
is the opposite of the crossover geometries in `hetero_rules_run.ipynb`, where over-amplifying
an already-sufficient drive destabilizes the trajectory.


In [ ]:
N_SWEEP = [20, 60, 100, 150, 200, 300, 400]

n_sweep_results = {
    tag: {n: single_sequence_score_sweep(catalogue, seq_len=SEQ_LEN,
                                         corruption_rate=CONFIG["corruption_rate"],
                                         N=n, trials=SINGLE_SEQ_TRIALS)
          for n in N_SWEEP}
    for tag, catalogue in (("raw", RAW_CATALOGUE), ("gain", GAIN_CATALOGUE))
}

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for row, tag in enumerate(("raw", "gain")):
    res = n_sweep_results[tag]
    for name in RAW_CATALOGUE:
        band(axes[row, 0], N_SWEEP, [res[n][name]["retrieval_fraction"] for n in N_SWEEP],
             [res[n][name]["fraction_std"] for n in N_SWEEP], label=name)
        band(axes[row, 1], N_SWEEP, [res[n][name]["mean_overlap"] for n in N_SWEEP],
             [res[n][name]["overlap_std"] for n in N_SWEEP], label=name)
    axes[row, 0].set_ylabel("retrieval fraction")
    axes[row, 0].set_title(f"Metric 1 -- retrieval fraction vs N -- {tag} (band = $\\pm 1$ s.d.)")
    axes[row, 1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
    axes[row, 1].set_ylabel("mean overlap (fully-retrieved trials; 0 = none)")
    axes[row, 1].set_title(f"Metric 3 -- overlap quality vs N -- {tag} (band = $\\pm 1$ s.d.)")
    for ax in axes[row]:
        ax.set_xlabel("N (number of neurons)")
        ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
fig.suptitle(f"2. Network-size sweep -- single smooth chain, P={SEQ_LEN + 1}, "
             f"corruption_rate={CONFIG['corruption_rate']} fixed")
plt.tight_layout(); plt.show()

for tag in ("raw", "gain"):
    for key, label in (("retrieval_fraction", "metric 1: retrieval fraction"),
                       ("retrieval_rate", "metric 2: strict full-retrieval rate")):
        print(f"-- {tag}, {label} --")
        print(f"{'rule':<15}" + "".join(f"N={n:<7}" for n in N_SWEEP))
        for name in RAW_CATALOGUE:
            print(f"{name:<15}" + "".join(f"{n_sweep_results[tag][n][name][key]:<8.0%}"
                                          for n in N_SWEEP))
        print()


## 2b. $L$-fraction against $N$, in the style of `kuramoto_sequence.ipynb`

Metric 1 is all-or-nothing. `kuramoto/sequence/kuramoto_sequence.ipynb` measures the same axis
with a partial-credit quantity instead --

$$\text{retrieval fraction} = \frac{\texttt{compare\_to\_stored\_sequence(retrieved, seq)}}{P}$$

the fraction of the chain recovered *in order* before the first mismatch -- sweeping $N$ with
the spread across generation seeds as error bars. That notebook uses it to find $L_{\max}(N)$
at a `retrieval_tolerance` of 0.9; here the length is fixed at `SEQ_LEN = 15` and $N$ is the
swept axis, so this is a vertical slice through that curve.

Left: mean retrieval fraction over `SEQ_SWEEP_SEEDS` seeds, $\pm 1$ s.d. Right: the fraction
of seeds clearing `RETRIEVAL_TOLERANCE`. This is the panel that shows the raw rules degrading
*gradually* with $N$ (0.93, 0.90, 0.53, 0.57 for `bigram`) rather than falling off a cliff --
which is what identifies weakening drive rather than an abrupt capacity limit.


In [ ]:
SEQ_N_SWEEP = [50, 100, 150, 200, 250, 300, 400, 500]
SEQ_SWEEP_SEEDS = 20                       # kuramoto_sequence.ipynb's seed_range
RETRIEVAL_TOLERANCE = 0.9                  # its SWEEP_CONFIG["retrieval_tolerance"]
RETRIEVAL_TOLERANCE_MARGIN = 0.05          # its SWEEP_CONFIG["retrieval_tolerance_margin"]


def retrieval_fraction(net, seq, duration):
    """The partial-credit measure of kuramoto_sequence.ipynb: the fraction of the stored
    sequence recovered, in order, before the first mismatch. 1.0 is full retrieval (and is
    exactly metric 1 succeeding); 0.4 means the chain broke 40% of the way in.

    Decoded with the peak decoder, for the reason given in the decoder note above -- the
    streaming decoder drops a pattern mid-chain and would report this fraction as the
    position of that drop rather than the position the trajectory actually reached."""
    result = net.simulate(seq, T=duration)
    retrieved = net.decode_sequence_peaks(result, seq, min_distance=PEAK_MIN_DISTANCE)
    return net.compare_to_stored_sequence(retrieved, seq) / len(seq)


def fraction_vs_N(catalogue, seq_len, N_values, seeds=SEQ_SWEEP_SEEDS, duration=None):
    """For each rule and each N, the retrieval fraction over `seeds` generation seeds.
    Every rule sees the same seeds at every N, so the curves are directly comparable."""
    duration = DWELL_PER_TRANSITION * seq_len if duration is None else duration
    out = {}
    for name, rule in catalogue.items():
        per_N = []
        for n in N_values:
            net = kc.ContextNetwork(rule=rule, N=n, seed=10, **CONFIG)
            fracs = [retrieval_fraction(net,
                                        krm.generate_sequence(n, seq_len, CONFIG["corruption_rate"],
                                                              name="A", seed=s),
                                        duration)
                     for s in range(seeds)]
            per_N.append(np.asarray(fracs))
        out[name] = per_N
    return out


frac_results = fraction_vs_N(RAW_CATALOGUE, SEQ_LEN, SEQ_N_SWEEP)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name, per_N in frac_results.items():
    means = [f.mean() for f in per_N]
    stds = [f.std() for f in per_N]
    axes[0].errorbar(SEQ_N_SWEEP, means, yerr=stds, marker="o", capsize=3, lw=1.6, label=name)
    axes[1].plot(SEQ_N_SWEEP, [(f >= RETRIEVAL_TOLERANCE).mean() for f in per_N], "-o", label=name)

axes[0].set_ylabel("retrieval fraction (recovered in order / $P$)")
axes[0].set_title(f"Retrieval fraction vs $N$ at $P={SEQ_LEN + 1}$ (bars = $\\pm 1$ s.d. over {SEQ_SWEEP_SEEDS} seeds)")
axes[1].axhline(0.5, color="k", ls=":", lw=1)
axes[1].set_ylabel(f"fraction of seeds with retrieval fraction $\\geq$ {RETRIEVAL_TOLERANCE}")
axes[1].set_title(f"Seeds clearing tolerance = {RETRIEVAL_TOLERANCE} "
                  f"$\\pm$ {RETRIEVAL_TOLERANCE_MARGIN}")
for ax in axes:
    ax.set_xlabel("N (number of neurons)")
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
fig.suptitle(f"2b. Partial-credit retrieval fraction vs N, seq_len={SEQ_LEN} "
             f"(kuramoto_sequence.ipynb's measure)")
plt.tight_layout(); plt.show()

print(f"mean retrieval fraction at P={SEQ_LEN + 1}, {SEQ_SWEEP_SEEDS} seeds")
print(f"{'rule':<15}" + "".join(f"N={n:<7}" for n in SEQ_N_SWEEP))
for name, per_N in frac_results.items():
    print(f"{name:<15}" + "".join(f"{f.mean():<9.2f}" for f in per_N))


---

## 2c. Scaling $\omega$ with $N$

§2 leaves a question: the raw context rules fail at large $N$ *at the configured*
`frequency_std = 0.3`. Is that about $N$, or about $\omega$? The overlaps concentrate as
$1/\sqrt{N}$ while $\omega_i \sim \mathcal{N}(0, \sigma_\omega)$ stays put, so the natural
thing to try is scaling the disorder down with $N$ to match:

$$\sigma_\omega(N) = \sigma_0 \left(\frac{N_0}{N}\right)^{p}, \qquad p \in \{0, \tfrac{1}{2}, 1\}$$

with $\sigma_0 = 0.3$ at $N_0 = 100$. $p = 0$ is §2's fixed-disorder sweep.

The grid below is the control that makes the answer unambiguous: it varies $N$ and
$\sigma_\omega$ **independently**, so a scaling law shows up as a tilted contour and a pure
$\sigma$ effect shows up as flat columns.


In [ ]:
OMEGA_SCALE_N = [50, 100, 200, 400]
OMEGA_SCALE_SIGMA = [0.05, 0.10, 0.15, 0.20, 0.30, 0.45]
OMEGA_SCALE_RULES = ["forward", "bigram", "coh bigram", "coh trigram"]
SIGMA_REF, N_REF = 0.3, 100          # the configured operating point
OMEGA_SCALE_TRIALS = 12


def fraction_at(rule_name, n, sigma, trials=OMEGA_SCALE_TRIALS):
    """Metric 1 for one rule at one (N, sigma_omega), frequency_mean = 0."""
    res = single_sequence_score_sweep(
        {rule_name: RAW_CATALOGUE[rule_name]}, seq_len=SEQ_LEN,
        corruption_rate=CONFIG["corruption_rate"], N=n, trials=trials,
        config_overrides={"frequency_mean": 0.0, "frequency_std": sigma})
    return res[rule_name]["retrieval_fraction"]


omega_grid = {name: np.array([[fraction_at(name, n, sig) for sig in OMEGA_SCALE_SIGMA]
                              for n in OMEGA_SCALE_N])
              for name in OMEGA_SCALE_RULES}

fig, axes = plt.subplots(1, len(OMEGA_SCALE_RULES), figsize=(5.0 * len(OMEGA_SCALE_RULES), 4.4))
for ax, name in zip(np.atleast_1d(axes), OMEGA_SCALE_RULES):
    grid = omega_grid[name]
    im = ax.imshow(grid, cmap="viridis", vmin=0.0, vmax=1.0, origin="lower", aspect="auto")
    for i in range(len(OMEGA_SCALE_N)):
        for j in range(len(OMEGA_SCALE_SIGMA)):
            ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="w" if grid[i, j] < 0.6 else "k")
    # the p = 1/2 scaling law, drawn on the grid for comparison
    xs = [np.interp(SIGMA_REF * (N_REF / n) ** 0.5, OMEGA_SCALE_SIGMA,
                    range(len(OMEGA_SCALE_SIGMA))) for n in OMEGA_SCALE_N]
    ax.plot(xs, range(len(OMEGA_SCALE_N)), "r--o", lw=1.5, ms=5,
            label=r"$\sigma_\omega \propto N^{-1/2}$")
    ax.set_xticks(range(len(OMEGA_SCALE_SIGMA)), [f"{s:g}" for s in OMEGA_SCALE_SIGMA])
    ax.set_yticks(range(len(OMEGA_SCALE_N)), [str(n) for n in OMEGA_SCALE_N])
    ax.set_xlabel(r"$\sigma_\omega$"); ax.set_ylabel("N")
    ax.set_title(f"{name} (raw) -- retrieval fraction", fontsize=10)
    ax.legend(fontsize=8, loc="lower left")
fig.suptitle(f"2c. Retrieval fraction over $N \\times \\sigma_\\omega$ (seq_len={SEQ_LEN}, "
             f"{OMEGA_SCALE_TRIALS} seeds)")
plt.tight_layout(); plt.show()

print(f"Metric 1 along the scaling laws, sigma(N) = {SIGMA_REF} * ({N_REF}/N)^p")
for p_exp in (0.0, 0.5, 1.0):
    print(f"-- p = {p_exp:g}   (sigma: " +
          ", ".join(f"N={n}:{SIGMA_REF * (N_REF / n) ** p_exp:.3f}" for n in OMEGA_SCALE_N) + ")")
    print(f"   {'rule':<13}" + "".join(f"N={n:<9}" for n in OMEGA_SCALE_N))
    for name in OMEGA_SCALE_RULES:
        row = "".join(f"{fraction_at(name, n, SIGMA_REF * (N_REF / n) ** p_exp):<11.2f}"
                      for n in OMEGA_SCALE_N)
        print(f"   {name:<13}{row}")
    print()


### What the grid says

Scaling $\sigma_\omega$ down with $N$ **does** restore the raw context rules -- at
$p = \tfrac{1}{2}$, `bigram` and `coh bigram` go to a retrieval fraction of 1.00 at
$N = 200$ and $400$, and `coh trigram` to 0.94 / 1.00, against 0.55 / 0.58 at fixed
$\sigma_\omega$. But the grid shows that is **not** a scaling law doing the work.

Read the columns: at $\sigma_\omega \le 0.2$ every rule scores 1.00 at every $N \ge 100$,
with no $N$-dependence at all. The $N$-dependence only appears in the $\sigma_\omega = 0.3$
column -- the configured value -- where it runs 0.89, 0.88, 0.55, 0.56 for `bigram`. At
$\sigma_\omega = 0.45$ everything fails everywhere.

So the raw rules have a **critical disorder** $\sigma_c$ between 0.2 and 0.3, and the default
$\sigma_\omega = 0.3$ sits right on that edge -- just inside it at small $N$, just outside at
large $N$. What §2 measured as a collapse with $N$ is that edge moving down slightly as the
overlaps concentrate; it is not a capacity limit, and $N$ is not the control parameter. The
$p = \tfrac{1}{2}$ curve works because it happens to cross from the $\sigma = 0.3$ column into
the $\sigma \le 0.2$ region, not because it matches any $1/\sqrt{N}$ in the dynamics.

Two ways to state the same conclusion, both visible in the grid:

- hold the drive and lower the disorder ($\sigma_\omega \le 0.2$), or
- hold the disorder and raise the drive (`proportional_gain`, §2),

and the raw context rules retrieve a 16-pattern chain at every $N$. They are running at the
edge of their disorder tolerance, not at the edge of their capacity.

$N = 50$ is the one row that does not reach 1.00 at any $\sigma_\omega$ (0.84-0.95): 16
patterns in 50 neurons is a genuinely loaded network, which is the capacity effect the rest of
the grid is *not*.


---

# 3. Superposition of learning rules

**Topology:** single smooth chain, `SEQ_LEN = 15`.  **Metrics:** 1, strict full-retrieval
success rate; 2, overlap quality (mean peak overlap) conditional on it, with its spread.

All `C(4,2) = 6` pairs among the four context rules (`bigram`, `coh bigram`, `coh trigram`,
`multilag`), mixed via `kc.mixture` at relative weight `w` (`w=0` is the pure first rule,
`w=1` the pure second). `multilag_rule` doesn't take a single `weight=` kwarg the way the
other three builders do -- its weight lives across three rows (one per lag), not one --
so the mixing coefficient is applied uniformly to a rule's *whole* weight column after
construction (`scaled`, below), rather than relying on each builder's own `weight=`
argument. This treats all four base rules the same way regardless of how many rows they
expand to.


In [ ]:
BASE_RULES = {
    "bigram":      kc.bigram_rule(),
    "coh bigram":  kc.coherent_bigram_rule(),
    "coh trigram": kc.coherent_trigram_rule(),
    "multilag":    kc.multilag_rule((1.0, 1.0, 1.0)),
}


def scaled(rule, w):
    """Copy of `rule` with its whole weight column multiplied by w -- works uniformly for
    single-row and multi-row rules, unlike passing weight= to the original builder."""
    rule = np.atleast_2d(np.asarray(rule, dtype=np.float64)).copy()
    rule[:, -1] *= w
    return rule


SUPERPOSITION_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]

superposition_results = {}
for name_a, name_b in itertools.combinations(BASE_RULES, 2):
    pair_label = f"{name_a} / {name_b}"
    per_weight = []
    for w in SUPERPOSITION_WEIGHTS:
        rule = kc.mixture(scaled(BASE_RULES[name_a], 1 - w), scaled(BASE_RULES[name_b], w))
        result = single_sequence_score_sweep({pair_label: rule}, seq_len=SEQ_LEN,
                                             corruption_rate=CONFIG["corruption_rate"],
                                             N=N, trials=SINGLE_SEQ_TRIALS)
        per_weight.append(result[pair_label])
    superposition_results[pair_label] = per_weight

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for pair_label, per_weight in superposition_results.items():
    band(axes[0], SUPERPOSITION_WEIGHTS, [r["retrieval_fraction"] for r in per_weight],
         [r["fraction_std"] for r in per_weight], label=pair_label)
    band(axes[1], SUPERPOSITION_WEIGHTS, [r["mean_overlap"] for r in per_weight],
         [r["overlap_std"] for r in per_weight], label=pair_label)
axes[0].set_xlabel("w  (0 = pure first rule, 1 = pure second rule)")
axes[0].set_ylabel("retrieval fraction")
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title("Metric 1 -- retrieval fraction across rule superpositions")
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_xlabel("w  (0 = pure first rule, 1 = pure second rule)")
axes[1].set_ylabel("mean overlap (fully-retrieved trials; 0 = none)")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Metric 3 -- overlap quality across superpositions (band = $\\pm 1$ s.d.)")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)
fig.suptitle(f"3. Rule superposition -- single smooth chain, P={SEQ_LEN + 1}, strict full retrieval")
plt.tight_layout(); plt.show()

print(f"{'pair':<28}" + "".join(f"w={w:<9g}" for w in SUPERPOSITION_WEIGHTS)
      + "   (retrieval fraction / strict rate)")
for pair_label, per_weight in superposition_results.items():
    print(f"{pair_label:<28}" + "".join(f"{r['retrieval_fraction']:.0%}/{r['retrieval_rate']:<6.0%}"
                                        for r in per_weight))


---

# 4. Crossover geometries

Every sweep above scored a single, non-branching sequence. The same correctness-first mean
overlap methodology extends to the two-sequence crossover experiments from
`hetero_rules_run.ipynb`: a cueing attempt only contributes to `mean_overlap` when
`classify_unshared` calls it `"correct"` (every *unshared* pattern of the cued branch
decoded, in order); a failed attempt only lowers `retrieval_rate`. Shared positions are
excluded from the overlap average since both branches store the identical pattern there --
it carries no rule-discriminating information.

Four before-shared-after geometries are tested -- the same set as `hetero_rules_run.ipynb`,
for comparability:

| tag | spec | $P$ | shared | $L$ | duration | reach needed |
|---|---|---|---|---|---|---|
| `2-1-2` | **(a)** eq (7) | 5 | $\{2\}$ | 1 | 100 | 1 -- `bigram`, `coh bigram` |
| `2-2-2` | **(c)** eq (9) | 6 | $\{2,3\}$ | 2 | 120 | 2 -- only `coh trigram` |
| `2-3-2` | failure case | 7 | $\{2,3,4\}$ | 3 | 140 | 3 -- nothing in the catalogue |
| `1-1-1` | legacy | 3 | $\{1\}$ | 1 | 60 | 1 |

Benchmark **(b)**, eq (8) -- three branches sharing one pattern -- is not implemented yet.
These sweeps keep the **streaming** decoder: what is scored is which branch the trajectory
commits to and in what order, which is a streaming decision.


In [ ]:
def make_crossover_multisequence3(N, corruption_rate, origin_seed, mutation_seeds,
                                  seq_len, shared_idx):
    """Two sequences of `seq_len` patterns sharing a contiguous run at `shared_idx`.
    Ported from hetero_rules_run.ipynb -- see that notebook for the full explanation."""
    shared_idx = sorted(shared_idx)
    lo, hi = shared_idx[0], shared_idx[-1]
    if shared_idx != list(range(lo, hi + 1)):
        raise ValueError("shared_idx must be a contiguous run.")
    if not (0 < lo and hi < seq_len - 1):
        raise ValueError("The run needs at least one unshared pattern on each side.")

    seeds = iter(mutation_seeds)
    mutate = lambda v: krm.corrupt_phase(v, corruption_rate, np.random.default_rng(next(seeds)))

    run = [np.random.default_rng(origin_seed).choice([0.0, np.pi], size=N)]
    for _ in range(hi - lo):
        run.append(mutate(run[-1]))

    multi = krm.MultiSequence()
    for name in ("A", "B"):
        prefix, v = [], run[0]
        for _ in range(lo):
            v = mutate(v); prefix.append(v)
        prefix.reverse()
        suffix, v = [], run[-1]
        for _ in range(seq_len - 1 - hi):
            v = mutate(v); suffix.append(v)
        multi.add(krm.sequence_from_patterns(name, prefix + list(run) + suffix))
    return multi


def classify_unshared(retrieved, multi, name, shared_idx):
    """METRIC 1 -- correct = all unshared positions of the cued branch decoded, in order.
    Ported from hetero_rules_run.ipynb -- see that notebook for the full explanation."""
    other = "B" if name == "A" else "A"
    shared = set(shared_idx)
    pending = iter([k for k in range(len(multi[name])) if k not in shared])
    target = next(pending, None)
    for seq, idx in retrieved:
        if target is None:
            break
        if idx in shared:
            continue
        if seq == other and idx >= target:
            return "crossed"
        if seq == name and idx == target:
            target = next(pending, None)
    return "correct" if target is None else "incomplete"


def crossover_score_sweep(catalogue, seq_len, shared_idx,
                          trials=SINGLE_SEQ_TRIALS, seed=0, duration=100.0, boundary=None):
    """Like single_sequence_score_sweep, but for a branching topology.

    retrieval_rate (metric 1) is the STRICT success rate -- the number of cueing attempts
    (both A and B, each counted separately) classified "correct" by `classify_unshared`,
    divided by the number of attempts.
    mean_overlap / overlap_std (metric 2) average, only over those successful attempts, the
    mean peak overlap of the cued branch's own unshared patterns."""
    rng = np.random.default_rng(seed)
    shared = set(shared_idx)
    results = {}
    for name, rule in catalogue.items():
        net = kc.ContextNetwork(rule=rule, N=N, seed=10, **CONFIG)
        overlap_vals, n_attempts = [], 0
        for _ in range(trials):
            origin_seed = int(rng.integers(SEED_POOL))
            mutation_seeds = rng.choice(SEED_POOL, size=12, replace=False).tolist()
            multi = make_crossover_multisequence3(N, CONFIG["corruption_rate"], origin_seed,
                                                  mutation_seeds, seq_len, shared_idx)
            labels = multi.labels()
            for cue_name in ("A", "B"):
                n_attempts += 1
                result = net.simulate(multi, cue=multi[cue_name], cue_idx=0, T=duration, boundary=boundary)
                retrieved = net.decode_sequence_online(result, multi, margin=MARGIN)
                if classify_unshared(retrieved, multi, cue_name, shared_idx) == "correct":
                    overlaps = net.overlaps(result["theta_history"], multi)
                    own_unshared = [labels.index((cue_name, k)) for k in range(seq_len) if k not in shared]
                    overlap_vals.append(float(overlaps[:, own_unshared].max(axis=0).mean()))
        results[name] = dict(
            retrieval_rate=len(overlap_vals) / n_attempts,
            n_correct=len(overlap_vals),
            mean_overlap=float(np.mean(overlap_vals)) if overlap_vals else float("nan"),
            overlap_std=float(np.std(overlap_vals)) if overlap_vals else float("nan"),
            overlap_var=float(np.var(overlap_vals)) if overlap_vals else float("nan"),
        )
    return results


# before-shared-after; all built as smooth (mutation-chain) sequences.
# Same table as hetero_rules_run.ipynb's GEOMETRIES.
CROSSOVER_GEOMETRIES = {
    "2-1-2": dict(seq_len=5, shared_idx=(2,),      duration=100.0),   # (a) eq (7)
    "2-2-2": dict(seq_len=6, shared_idx=(2, 3),    duration=120.0),   # (c) eq (9)
    "2-3-2": dict(seq_len=7, shared_idx=(2, 3, 4), duration=140.0),   # the failure case
    "1-1-1": dict(seq_len=3, shared_idx=(1,),      duration=60.0),    # legacy
}

crossover_results = {tag: crossover_score_sweep(RAW_CATALOGUE, **geom)
                     for tag, geom in CROSSOVER_GEOMETRIES.items()}

for tag in CROSSOVER_GEOMETRIES:
    print(f"-- {tag} --")
    for name, r in crossover_results[tag].items():
        q = (f"{r['mean_overlap']:+.3f} +/- {r['overlap_std']:.3f}"
             if r["mean_overlap"] == r["mean_overlap"] else "n/a (none correct)")
        print(f"  {name:<15} success={r['retrieval_rate']:>5.0%}   overlap={q}")

tags = list(CROSSOVER_GEOMETRIES)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for name in RAW_CATALOGUE:
    axes[0].plot(tags, [crossover_results[t][name]["retrieval_rate"] for t in tags], "-o", label=name)
    band(axes[1], tags, [crossover_results[t][name]["mean_overlap"] for t in tags],
         [crossover_results[t][name]["overlap_std"] for t in tags], label=name)
axes[0].set_ylabel("full-retrieval success rate (unshared patterns)")
axes[0].set_title("Metric 1 -- crossover success by topology")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean overlap (successful attempts only; 0 = none)")
axes[1].set_title("Metric 2 -- crossover overlap quality (band = $\\pm 1$ s.d.)")
for ax in axes:
    ax.set_xlabel("topology (before-shared-after)")
    ax.set_ylim(-0.05, 1.05); ax.tick_params(axis="x", rotation=20)
    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
fig.suptitle("4. Crossover geometries -- strict full retrieval of the cued branch's unshared patterns")
plt.tight_layout(); plt.show()


---

# 5. Settle vs. loop (`boundary="self"` vs `"cycle"`)

`net.simulate(..., boundary=...)` controls what the last stored pattern drives: `"self"`
(the default used everywhere above) clamps, so the terminal pattern drives itself and the
trajectory settles there; `"cycle"` wraps, so the terminal pattern drives the first one and
the sequence is meant to loop indefinitely. Both `single_sequence_score_sweep` and
`crossover_score_sweep` accept `boundary=` directly (threaded straight through to
`net.simulate`), so the comparison is just two calls at each geometry.


In [ ]:
for boundary in ("self", "cycle"):
    single = single_sequence_score_sweep(RAW_CATALOGUE, seq_len=SEQ_LEN,
                                         corruption_rate=CONFIG["corruption_rate"], N=N,
                                         boundary=boundary)
    cross = crossover_score_sweep(RAW_CATALOGUE, **CROSSOVER_GEOMETRIES["2-1-2"],
                                  boundary=boundary)
    print(f"boundary={boundary!r}")
    print(f"  {'rule':<15}{'single-seq fraction':>21}{'single-seq strict':>19}{'crossover 2-1-2':>18}")
    for name in RAW_CATALOGUE:
        print(f"  {name:<15}{single[name]['retrieval_fraction']:>20.0%}"
              f"{single[name]['retrieval_rate']:>19.0%}{cross[name]['retrieval_rate']:>18.0%}")
    print()


---

# 6. Frequency conditions

Two edge cases in the intrinsic-frequency disorder `omega`:

- **`frequency_mean=0, frequency_std=0`** -- no disorder at all. An exact, uncorrupted cue
  has `theta` in `{0, pi}` everywhere, so `sin(theta)=0`, and with `omega=0` too the drive
  term vanishes identically -- the trajectory is frozen at the cue forever (this is exactly
  why `CONFIG["frequency_std"] > 0` is asserted everywhere else in these notebooks). To make
  this configuration testable at all, the cue itself is displaced with a corruption that is
  *not* part of the stored sequence -- a copy of the first pattern with some neurons flipped,
  via `displaced_cue`, bypassing `net.simulate`'s normal (always uncorrupted) cue
  construction. **One correction to that:** flipping neurons is not enough on its own.
  `corrupt_phase` keeps every phase on the $\{0,\pi\}$ lattice, where $\sin\theta = 0$
  *exactly*, so a flip-only displacement is still an exact fixed point at $\omega = 0$ and
  the state does not move. `CUE_PHASE_JITTER` adds a small angular nudge on every neuron,
  taking $\theta$ off the lattice so the dynamics can run. The displaced state is never added
  to the sequence and never memorised -- it is only $\theta(0)$.
- **`frequency_mean=1, frequency_std=0`** -- uniform frequency, still no disorder, but
  nonzero. Cued normally (no displacement needed): at the exact cue, `sin(theta)=0` but
  `omega=1 != 0`, so the trajectory leaves the cue immediately regardless.


In [ ]:
CUE_DISPLACEMENT_RATE = CONFIG["corruption_rate"]   # neurons flipped off xi^0
CUE_PHASE_JITTER      = 0.2                         # radians, uniform on [-j, +j]

OMEGA_ZERO   = {"frequency_mean": 0.0, "frequency_std": 0.0}
OMEGA_ONE    = {"frequency_mean": 1.0, "frequency_std": 0.0}

freq_zero_results = single_sequence_score_sweep(
    RAW_CATALOGUE, seq_len=SEQ_LEN, corruption_rate=CONFIG["corruption_rate"], N=N,
    config_overrides=OMEGA_ZERO,
    cue_corruption_rate=CUE_DISPLACEMENT_RATE, cue_phase_jitter=CUE_PHASE_JITTER,
)

freq_one_results = single_sequence_score_sweep(
    RAW_CATALOGUE, seq_len=SEQ_LEN, corruption_rate=CONFIG["corruption_rate"], N=N,
    config_overrides=OMEGA_ONE,
)

def report(title, results):
    print(title)
    for name, r in results.items():
        q = (f"{r['mean_overlap']:+.3f} +/- {r['overlap_std']:.3f}"
             if r["mean_overlap"] == r["mean_overlap"] else "n/a (none correct)")
        print(f"  {name:<15} fraction={r['retrieval_fraction']:>5.0%}  "
              f"strict={r['retrieval_rate']:>5.0%}   overlap={q}")

report(f"(a) omega = 0, omega_std = 0 -- started from a displacement of xi^0 "
       f"({CUE_DISPLACEMENT_RATE:.0%} flips + {CUE_PHASE_JITTER} rad jitter; not a stored pattern)",
       freq_zero_results)
print()
report("(b) omega = 1, omega_std = 0 -- exact cue, no displacement needed", freq_one_results)

names = list(RAW_CATALOGUE)
x = np.arange(len(names))
CASES = [(freq_zero_results, r"$\omega \equiv 0$ (displaced start)", -0.2),
         (freq_one_results,  r"$\omega \equiv 1$ (exact cue)",       +0.2)]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for results, label, off in CASES:
    axes[0].bar(x + off, [results[n]["retrieval_fraction"] for n in names], 0.4, label=label,
                yerr=[results[n]["fraction_std"] for n in names], capsize=2)
    axes[1].bar(x + off, zeroed([results[n]["mean_overlap"] for n in names]), 0.4, label=label,
                yerr=zeroed([results[n]["overlap_std"] for n in names]), capsize=2)
axes[0].set_ylabel("retrieval fraction")
axes[0].set_title(f"6. Metric 1 -- retrieval fraction, P={SEQ_LEN + 1}")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean overlap (fully-retrieved trials; 0 = none)")
axes[1].set_title("6. Metric 3 -- overlap quality (bars = $\\pm 1$ s.d.)")
for ax in axes:
    ax.set_xticks(x, names, rotation=20, ha="right", fontsize=9)
    ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


### The retrieval time series from a displaced start ($\omega \equiv 0$)

One run per rule. The trajectory starts from the displacement (dotted line at $t = 0$ -- note
it does not begin at overlap 1, because the start is *off* $\xi^0$), pulls in onto $\xi^0$,
then walks the chain. With no frequency disorder, how far the peaks stay above `tolerance`
before the cascade runs out of drive is what caps the success rate above.


In [ ]:
OMEGA_ZERO_EXAMPLE_RULES = ["forward", "bigram", "coh bigram", "coh trigram"]
OMEGA_ZERO_EXAMPLE_SEED = 0

fig, axes = plt.subplots(len(OMEGA_ZERO_EXAMPLE_RULES), 1,
                         figsize=(11, 3.4 * len(OMEGA_ZERO_EXAMPLE_RULES)), sharex=True)
duration = DWELL_PER_TRANSITION * SEQ_LEN

for ax, name in zip(axes, OMEGA_ZERO_EXAMPLE_RULES):
    net = kc.ContextNetwork(rule=RAW_CATALOGUE[name], N=N, seed=10, **{**CONFIG, **OMEGA_ZERO})
    rng = np.random.default_rng(OMEGA_ZERO_EXAMPLE_SEED)
    seq = krm.generate_sequence(N, SEQ_LEN, CONFIG["corruption_rate"], name="A",
                                seed=OMEGA_ZERO_EXAMPLE_SEED)
    result = run_single_trial(net, seq, duration,
                              cue_corruption_rate=CUE_DISPLACEMENT_RATE,
                              cue_phase_jitter=CUE_PHASE_JITTER, rng=rng)
    retrieved = net.decode_sequence_online(result, seq, margin=MARGIN)
    reached = net.compare_to_stored_sequence(retrieved, seq)

    net.plot(result, seq, ax=ax,
             title=f'"{name}"  --  $\\omega \\equiv 0$, started from a displacement of $\\xi^0$'
                   f'   (in-order patterns recovered: {reached}/{len(seq)})')
    ax.axhline(CONFIG["tolerance"], color="k", ls="--", lw=1)
    ax.axhline(result["overlaps"][0].max(), color="tab:grey", ls=":", lw=1)
    ax.set_ylim(-1.05, 1.05)
    ax.legend(fontsize=6, ncol=6)

fig.suptitle(r"6(a). Retrieval time series at $\omega \equiv 0$, $\sigma_\omega = 0$ "
             f"(displaced start, not a stored pattern; P={SEQ_LEN + 1})", y=1.001)
plt.tight_layout(); plt.show()


---

# 7. Random (uncorrelated) patterns instead of a mutation chain

Every sequence tested so far is a mutation chain: pattern $\mu{+}1$ is a corrupted copy of
pattern $\mu$, so consecutive patterns are always similar by construction, and the corruption-ratio
sweep above varies exactly *how* similar. This section asks the more basic question that
sweep assumes an answer to: does retrieval depend on that similarity at all? Here
the stored sequence is `seq_len+1` **independently random** patterns
(`generate_random_sequence`) -- no relationship between consecutive patterns whatsoever.
20 seeds, as requested.

Run for **both** catalogues, since section 2 showed the gain correction is what separates
"under-driven" from "cannot do it". Here it changes nothing: peak overlaps sit at 0.16-0.19
against `tolerance = 0.5` either way. On a smooth chain the raw context rules have the right
drive direction and too little of it; on uncorrelated patterns there is no overlap for the
drive to build on at all, and scaling a gate that reads $m \approx 0$ scales nothing.


In [ ]:
RANDOM_TRIALS = 20

random_results = {
    tag: single_sequence_score_sweep(catalogue, seq_len=SEQ_LEN,
                                     corruption_rate=CONFIG["corruption_rate"],
                                     N=N, trials=RANDOM_TRIALS, random_patterns=True)
    for tag, catalogue in (("raw", RAW_CATALOGUE), ("gain", GAIN_CATALOGUE))
}

# highest overlap reached on any non-cue pattern, regardless of outcome -- with 0% success
# everywhere, metric 2 is undefined and this is what says how far off the rules actually are
def best_reached(catalogue, trials=RANDOM_TRIALS, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name, rule in catalogue.items():
        net = kc.ContextNetwork(rule=rule, N=N, seed=10, **CONFIG)
        vals = []
        for _ in range(trials):
            seq = generate_random_sequence(N, SEQ_LEN, name="A", seed=int(rng.integers(SEED_POOL)))
            result = net.simulate(seq, T=DWELL_PER_TRANSITION * SEQ_LEN)
            vals.append(float(result["overlaps"][:, 1:].max(axis=0).mean()))
        out[name] = float(np.mean(vals))
    return out

reached = {tag: best_reached(cat) for tag, cat in (("raw", RAW_CATALOGUE), ("gain", GAIN_CATALOGUE))}

print(f"Random (uncorrelated) patterns, P={SEQ_LEN + 1}, {RANDOM_TRIALS} seeds "
      f"(tolerance = {CONFIG['tolerance']})")
print(f"{'rule':<15}{'raw frac':>10}{'raw peak':>10}{'gain frac':>11}{'gain peak':>11}")
for name in RAW_CATALOGUE:
    print(f"{name:<15}{random_results['raw'][name]['retrieval_fraction']:>10.0%}{reached['raw'][name]:>10.2f}"
          f"{random_results['gain'][name]['retrieval_fraction']:>11.0%}{reached['gain'][name]:>11.2f}")

names = list(RAW_CATALOGUE)
x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for off, tag in ((-0.2, "raw"), (0.2, "gain")):
    axes[0].bar(x + off, [random_results[tag][n]["retrieval_fraction"] for n in names], 0.4, label=tag)
    axes[1].bar(x + off, [reached[tag][n] for n in names], 0.4, label=tag)
axes[0].set_ylabel("retrieval fraction")
axes[0].set_title("Metric 1 -- retrieval fraction, random patterns")
axes[1].axhline(CONFIG["tolerance"], color="k", ls="--", lw=1, label="tolerance")
axes[1].set_ylabel("mean best overlap reached (all trials)")
axes[1].set_title("How far off? mean peak overlap per pattern")
for ax in axes:
    ax.set_xticks(x, names, rotation=20, ha="right"); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
fig.suptitle(f"7. Independently random patterns (seq_len={SEQ_LEN}, {RANDOM_TRIALS} seeds)")
plt.tight_layout(); plt.show()


### A few example trajectories

`retrieval_rate=0%` across the board is a strong claim -- worth seeing directly rather than
only as a number. Below: one random-pattern sequence each for a spread of rules (simplest
to most context-heavy), cued from `seq[0]`, plotting overlap with every stored pattern over
time.


In [ ]:
EXAMPLE_RULES = ["forward", "bigram", "coh trigram"]

fig, axes = plt.subplots(len(EXAMPLE_RULES), 1, figsize=(11, 3.6 * len(EXAMPLE_RULES)), sharex=True)
for ax, name in zip(axes, EXAMPLE_RULES):
    net = kc.ContextNetwork(rule=RAW_CATALOGUE[name], N=N, seed=10, **CONFIG)
    seq = generate_random_sequence(N, seq_len=SEQ_LEN, name="A", seed=0)
    result = net.simulate(seq, T=DWELL_PER_TRANSITION * SEQ_LEN)
    net.plot(result, seq, ax=ax, title=f'"{name}" -- random patterns, cued from A[0]')
    ax.axhline(CONFIG["tolerance"], color="k", ls="--", lw=1)
    ax.set_ylim(-1.05, 1.05)
    ax.legend(fontsize=6, ncol=6)
plt.tight_layout(); plt.show()


---

# Findings

*(to be filled in from the executed results above)*

## Where to change things

- **Which rules are compared** -- `RAW_CATALOGUE`.
- **Sequence length / integration time** -- `SEQ_LEN` and `DWELL_PER_TRANSITION`; duration
  is always `DWELL_PER_TRANSITION * seq_len` unless a call passes `duration=` explicitly.
- **Crossover topologies** -- `CROSSOVER_GEOMETRIES` (`seq_len`, `shared_idx`, `duration`).
  The shared run must be contiguous with at least one unshared pattern on each side.
- **The displaced start** -- `CUE_DISPLACEMENT_RATE` and `CUE_PHASE_JITTER`. The jitter must
  stay > 0 at $\omega \equiv 0$ or the start is an exact fixed point.
- **Statistics** -- `SINGLE_SEQ_TRIALS`.
